# Exploring Research Papers with RAG

##Introduction
The rapid growth of scientific literature has made information retrieval increasingly challenging. Relevant findings are often scattered across lengthy documents and expressed using specialized terminology, making traditional keyword search insufficient for many research tasks. Retrieval-Augmented Generation (RAG) addresses this problem by combining document retrieval with language models, allowing relevant information to be located and used more effectively. Techniques such as BGE embeddings, vector search, and hybrid retrieval help identify both semantic relationships and exact technical terms within scientific texts. These capabilities support faster literature exploration, improved knowledge discovery, and more efficient analysis of large collections of research documents. The objective of this project is to build a  RAG retrieval system that processes research documents, indexes their content using semantic embeddings and keyword-based methods, and retrieves the most relevant passages in response to natural language queries.


##System Architecture
```
PDF/TXT Input (PyMuPDF / Tesseract)  
          ↓
    Text Extraction
          ↓
    Chunking
          ↓
    Embeddings (BGE)
          ↓
Qdrant (Vector), BM25 (Keyword)
          ↓
     Hybrid Search (RRF)
          ↓
   Ranked Results
```

In [ ]:
!pip install pymupdf pytesseract pillow nltk langchain-text-splitters

In [ ]:
from dataclasses import dataclass
from typing import Dict


@dataclass
class DocumentChunk:
  chunk_id: str
  text: str
  metadata: Dict

In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io


class PDFParser:

  def extract_text(self, pdf_path: str):
      doc = fitz.open(pdf_path)

      pages = []

      for page_number in range(len(doc)):
          page = doc[page_number]

          text = page.get_text("text").strip()

          # OCR fallback if no text found
          if not text:
              pix = page.get_pixmap(dpi=300)

              img_bytes = pix.tobytes("png")
              image = Image.open(io.BytesIO(img_bytes))

              text = pytesseract.image_to_string(image)

          pages.append({
              "page": page_number + 1,
              "text": text
          })

      return pages

In [ ]:
class TextParser:

  def extract_text(self, file_path: str):

      with open(file_path, "r", encoding="utf-8") as f:
          text = f.read()

      return [{
          "page": 1,
          "text": text
      }]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid
from typing import List

class TextChunker:

    def __init__(self, chunk_size=800, overlap=150):

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
            separators=["\n\n", "\n", ".", " "]
        )

    def chunk_text(self, pages) -> List[DocumentChunk]:

        chunks = []

        for page in pages:

            split_chunks = self.splitter.split_text(page["text"])

            for chunk in split_chunks:

                chunks.append(
                    DocumentChunk(
                        chunk_id=str(uuid.uuid4()),
                        text=chunk,
                        metadata={
                            "page": page["page"]
                        }
                    )
                )

        return chunks

In [ ]:
from pathlib import Path

#from pdf_parser import PDFParser
#from text_parser import TextParser


class DocumentLoader:

  def __init__(self):
      self.pdf_parser = PDFParser()
      self.text_parser = TextParser()

  def load(self, file_path: str):

      ext = Path(file_path).suffix.lower()

      if ext == ".pdf":
          return self.pdf_parser.extract_text(file_path)

      elif ext == ".txt":
          return self.text_parser.extract_text(file_path)

      else:
          raise ValueError(f"Unsupported file type: {ext}")

In [ ]:
!wget https://arxiv.org/pdf/2603.00096 ; mv 2603.00096 sample.pdf

--2026-06-08 10:51:42--  https://arxiv.org/pdf/2603.00096
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.131.42, 151.101.3.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 858529 (838K) [application/pdf]
Saving to: ‘2603.00096’

2603.00096          100%[===================>] 838.41K  --.-KB/s    in 0.03s   

2026-06-08 10:51:42 (23.4 MB/s) - ‘2603.00096’ saved [858529/858529]



In [ ]:
import json
#from ingest.loader import DocumentLoader
#from ingest.chunker import TextChunker

def main():

  file_path = "sample.pdf"

  loader = DocumentLoader()
  pages = loader.load(file_path)

  chunker = TextChunker(
      chunk_size=500,
      overlap=100
  )

  chunks = chunker.chunk_text(pages)

  output = []

  for chunk in chunks:
      output.append({
          "chunk_id": chunk.chunk_id,
          "text": chunk.text,
          "metadata": chunk.metadata
      })

  with open("chunks.json", "w") as f:
      json.dump(output, f, indent=2)

  print(f"Generated {len(chunks)} chunks")


if __name__ == "__main__":
  main()

Generated 236 chunks


In [ ]:
#Load JSON
import json

with open("chunks.json", "r") as f:
    data = json.load(f)

print(data)

[{'chunk_id': 'fdb3040c-34a2-40ce-bd68-b8055200a564', 'text': 'Particle acceleration to PeV energies in Pulsar Wind Nebula: a two zone model\nGunindra Krishna Mahanta\n \n \na,b, Nilay Bhatta, Bitan Ghosala,b, Subir Bhattacharyyaa,b\naAstrophysical Sciences Division, Bhabha Atomic Research Centre, Mumbai, 400085, Maharashtra, India\nbHomi Bhabha National Institute, Anushaktinagar, Mumbai, 400094, Maharashtra, India\nAbstract\nPeVatrons are the extreme galactic accelerators capable of producing PeV particles. Recent observation of Large High Altitude Air', 'metadata': {'page': 1}}, {'chunk_id': '84e856d9-808d-48b4-8850-faa93d65bd6e', 'text': 'Shower Observatory have detected UHE photons (≥100 TeV) from 43 galactic sources. Detection of UHE photons demands the\npresence of at least PeV particles in the acceleration site. Although the exact nature of most of the sources are still unknown, a large\nfraction of these sources have spatial association with pulsar wind nebula. In this work we 

In [ ]:
data[0]

{'chunk_id': 'fdb3040c-34a2-40ce-bd68-b8055200a564',
 'text': 'Particle acceleration to PeV energies in Pulsar Wind Nebula: a two zone model\nGunindra Krishna Mahanta\n \n \na,b, Nilay Bhatta, Bitan Ghosala,b, Subir Bhattacharyyaa,b\naAstrophysical Sciences Division, Bhabha Atomic Research Centre, Mumbai, 400085, Maharashtra, India\nbHomi Bhabha National Institute, Anushaktinagar, Mumbai, 400094, Maharashtra, India\nAbstract\nPeVatrons are the extreme galactic accelerators capable of producing PeV particles. Recent observation of Large High Altitude Air',
 'metadata': {'page': 1}}



---



In [ ]:
# scientific_rag_step1.py
# ============================================================
# Scientific RAG Retrieval Layer
# ============================================================
# Features:
# - Load chunks.json
# - Generate embeddings
# - Store in Qdrant
# - Semantic search
# - BM25 keyword retrieval
# - Hybrid search (RRF)
# - FastAPI API
#
# ============================================================
# INSTALL
# ============================================================
# pip install sentence-transformers qdrant-client \
#             rank-bm25 fastapi uvicorn numpy
#
# Start Qdrant:
# docker run -p 6333:6333 qdrant/qdrant
#
# Run:
# python scientific_rag_step1.py
#
# Start API:
# uvicorn scientific_rag_step1:app --reload
#
# ============================================================



In [ ]:
!pip install sentence-transformers qdrant-client rank-bm25 fastapi uvicorn > /dev/null 2>&1

In [ ]:
import json
import uuid
import numpy as np

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    PointStruct,
)

from fastapi import FastAPI

In [ ]:
# ============================================================
# CONFIG
# ============================================================

COLLECTION_NAME = "scientific_rag"

CHUNKS_PATH = "chunks.json"

EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

TOP_K = 5


In [ ]:
# ============================================================
# LOAD CHUNKS Reads your chunks.json and extracts the text chunks created during ingestion.
# ============================================================

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [chunk["text"] for chunk in chunks]


In [ ]:
len(texts)

248

In [ ]:
# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

model = SentenceTransformer(EMBEDDING_MODEL)


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
# ============================================================
# GENERATE EMBEDDINGS Embedding Model
# Converts each chunk into dense numerical vectors capturing semantic meaning.
# ============================================================

print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)


Generating embeddings...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
#  Qdrant Setup + Storage → Creates a vector database collection and stores embeddings with metadata.
# ============================================================
# CONNECT TO QDRANT
# ============================================================

#client = QdrantClient(
#    host="localhost",
#    port=6333,
#)


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

client = QdrantClient(":memory:")

In [ ]:
# ============================================================
# CREATE COLLECTION IF NOT EXISTS
# ============================================================

existing_collections = [
  c.name for c in client.get_collections().collections
]

if COLLECTION_NAME not in existing_collections:

  print("Creating Qdrant collection...")

  client.create_collection(
      collection_name=COLLECTION_NAME,
      vectors_config=VectorParams(
          size=len(embeddings[0]),
          distance=Distance.COSINE,
      ),
  )

Creating Qdrant collection...


In [ ]:
# ============================================================
# STORE EMBEDDINGS
# ============================================================

print("Uploading embeddings to Qdrant...")

points = []

for chunk, embedding in zip(chunks, embeddings):

  point = PointStruct(
      id=str(uuid.uuid4()),
      vector=embedding.tolist(),
      payload={
          "text": chunk["text"],
          "source": chunk.get("source", "unknown"),
          "page": chunk.get("page", -1),
          "chunk_id": chunk.get("id", "unknown"),
      },
  )

  points.append(point)

client.upsert(
  collection_name=COLLECTION_NAME,
  points=points,
)

print("Embeddings stored successfully.")


Uploading embeddings to Qdrant...
Embeddings stored successfully.


In [ ]:
# ============================================================
# BUILD BM25 INDEX
# ============================================================

print("Building BM25 index...")

tokenized_corpus = [
  text.lower().split()
  for text in texts
]

bm25 = BM25Okapi(tokenized_corpus)


Building BM25 index...


In [ ]:

# ============================================================
# SEMANTIC SEARCH
# ============================================================

def semantic_search(query, top_k=5):

  query_embedding = model.encode(
      query,
      normalize_embeddings=True,
  ).tolist()

  results = client.query_points(
      collection_name=COLLECTION_NAME,
      query=query_embedding,
      limit=top_k,
  )

  formatted_results = []

  for rank, result in enumerate(results):
      #print("rank ",rank)
      #print ("result",result)
      point, scorepoint = result
      scorepoint = scorepoint[0]
      #print("scorepoint ",scorepoint.id)

      formatted_results.append({
          "rank": rank + 1,
          "score": scorepoint.score,
          "text": scorepoint.payload["text"],
          "source": scorepoint.payload["source"],
          "page": scorepoint.payload["page"],
      })

  return formatted_results


In [ ]:
# @title
# ============================================================
# KEYWORD SEARCH
# ============================================================

def keyword_search(query, top_k=5):

  tokenized_query = query.lower().split()

  scores = bm25.get_scores(tokenized_query)

  top_indices = np.argsort(scores)[::-1][:top_k]

  results = []

  for rank, idx in enumerate(top_indices):

      results.append({
          "rank": rank + 1,
          "score": float(scores[idx]),
          "text": chunks[idx]["text"],
          "source": chunks[idx].get("source", "unknown"),
          "page": chunks[idx].get("page", -1),
      })

  return results


In [ ]:
# @title
# ============================================================
# HYBRID SEARCH (RRF)
# ============================================================

def hybrid_search(
    query,
    top_k=5,
    rrf_k=60,
):

    semantic_results = semantic_search(query, top_k)

    keyword_results = keyword_search(query, top_k)

    fusion_scores = {}

    # Semantic contribution
    for rank, result in enumerate(semantic_results):

        text = result["text"]

        fusion_scores[text] = (
            fusion_scores.get(text, 0)
            + 1 / (rrf_k + rank + 1)
        )

    # Keyword contribution
    for rank, result in enumerate(keyword_results):

        text = result["text"]

        fusion_scores[text] = (
            fusion_scores.get(text, 0)
            + 1 / (rrf_k + rank + 1)
        )

    reranked = sorted(
        fusion_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    final_results = []

    for rank, (text, score) in enumerate(reranked[:top_k]):

        final_results.append({
            "rank": rank + 1,
            "fusion_score": score,
            "text": text,
        })

    return final_results

In [ ]:
# ============================================================
# EXAMPLE SEARCH
# ============================================================

if __name__ == "__main__":

    query = "Why do the authors believe pulsar wind nebulae can produce PeV particles?"

    print("\n" + "=" * 60)
    print("SEMANTIC SEARCH")
    print("=" * 60)

    semantic_results = semantic_search(query)
    #print (semantic_results)
    for result in semantic_results:
        print ("QUERY:",query)
        print(f"\nScore: {result['score']:.4f}")
        print(f"Source: {result['source']} "
            f"| Page: {result['page']}"
        )
        print(result["text"][:300])
#---------------------------------------------
    print("\n" + "=" * 60)
    print("KEYWORD SEARCH")
    print("=" * 60)

    keyword_results = keyword_search(query)
    for result in keyword_results:
        rank = result['rank']
        if rank==1:
          print(f"\nScore: {result['score']:.4f}")
          print(
              f"Source: {result['source']} "
              f"| Page: {result['page']}"
          )
          print(result["text"][:300])
#-------------------------------------------------
    print("\n" + "=" * 60)
    print("HYBRID SEARCH")
    print("=" * 60)

    hybrid_results = hybrid_search(query)
    #print (hybrid_results)
    for result in hybrid_results:
        rank = result['rank']
        if rank==1:
          print(
              f"\nFusion Score: "
              f"{result['fusion_score']:.4f}"
          )

          print(result["text"][:300])


SEMANTIC SEARCH
QUERY: Why do the authors believe pulsar wind nebulae can produce PeV particles?

Score: 0.8517
Source: unknown | Page: -1
magnetization parameter σ only. Considering the eﬀect of σ, we show that in low σ environment pulsar wind nebula can produce
PeV particles. We have also investigate the role of turbulence in the nebular region in acceleration of particle to PeV energy. Current
study shows that both low σ environment

KEYWORD SEARCH

Score: 13.8332
Source: unknown | Page: -1
magnetization parameter σ only. Considering the eﬀect of σ, we show that in low σ environment pulsar wind nebula can produce
PeV particles. We have also investigate the role of turbulence in the nebular region in acceleration of particle to PeV energy. Current
study shows that both low σ environment

HYBRID SEARCH

Fusion Score: 0.0328
magnetization parameter σ only. Considering the eﬀect of σ, we show that in low σ environment pulsar wind nebula can produce
PeV particles. We have also investiga

##Evaluation by Chatgpt:

1–2 line evaluation (simple language)
- Semantic Search: Correctly finds that low σ + turbulence help produce PeV particles understands meaning, not just words.
- Keyword Search: Matches exact words like “σ”, “PeV”, and “turbulence” but does not really understand the idea behind them.
- Hybrid Search: Finds the right text but does not improve over semantic search; combination is not helping much.

####Comparison Table

|Search Type | Score /5 | Notes |
|--|--|--|
|Semantic Search|	4.5|	Understands idea: low magnetization + turbulence → PeV particles.|
|Keyword Search|	3.5 |	Finds exact words but lacks deeper understanding.|
|Hybrid Search|	3.0|	Gets correct text but fusion does not improve results.|

###Conclusion
The project implemented a Scientific RAG retrieval system combining semantic search, keyword search, and hybrid fusion for querying research documents. Results indicate that semantic retrieval was most reliable for capturing meaning, while keyword matching mainly supported exact term lookup. The weaker performance of hybrid fusion suggests limitations in score balancing and retrieval signal alignment. Future improvements could include better normalization, adaptive fusion strategies, and reranking to enhance overall retrieval quality.
